In [4]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# (F)(G) 設定
# =========================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
out_root  = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
model_name_dir = "RandomForest_Voc_oldFeatLogic"
os.makedirs(out_root, exist_ok=True)

file_names  = ["m_base_FP_rebuilt.csv"]
target_vars = ["Voc"]

# ★旧コードと同じく 4 目的変数をまとめて扱う
TARGET_VARS_ALL = ["Jsc", "Voc", "FF", "PCEmax"]

# (D)(E) 除外変数
INAPPROPRIATE_VARS = [
    "Jsccal", "JscJsccal", "PCEaTA", "PCEaTAFinal", "EQEABEST", "Rshthin", "Dresistance",
    "mobilityeblendOFET", "mobilityep1MOFET", "mobilityhblendSCLC", "mobilityhnMSCLC",
    "mobilityhp1MSCLC", "IonIoffF", "DRMS", "Var.1"
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

# =========================
# Helpers
# =========================
def safe_r2(y_true, y_pred) -> float:
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return float("nan")

def remove_collinear_features(df: pd.DataFrame, cols, threshold=0.99999):
    if len(cols) < 2:
        return cols
    corr = df[cols].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    return [c for c in cols if c not in to_drop]

def compute_importance(best_model, feature_cols):
    importances = best_model.feature_importances_
    rows = []
    for idx, val in enumerate(importances):
        if val > 0:
            rows.append({
                "Feature": feature_cols[idx],
                "Importance": float(val)
            })
    return rows

def flag_high_error(df_pred, quantile_prob=0.9):
    thr = float(df_pred["AbsError"].quantile(quantile_prob))
    df_pred["HighErrorFlag"] = df_pred["AbsError"] >= thr
    df_pred["HighErrorThreshold"] = thr
    return df_pred, thr

# =========================
# Main
# =========================
summary_rows    = []
importance_rows = []

for file in file_names:
    in_path = os.path.join(base_path, file)
    if not os.path.exists(in_path):
        print(f"[WARN] File not found: {in_path}")
        continue

    print(f"\n=== Processing Random Forest Voc (oldFeatLogic): {file} ===")
    df_raw = pd.read_csv(in_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()

    if df_num.shape[0] < 20:
        print(f"[WARN] Too few rows after dropna: {df_num.shape[0]}")
        continue

    for target in target_vars:
        if target not in df_num.columns:
            print(f"[WARN] Target {target} not in columns of {file}")
            continue

        # -----------------------------
        # (C)(D)(E) 特徴量選択（旧ロジック準拠）
        # -----------------------------
        # 4 目的変数を全部除外し、それ以外の数値列を候補にする
        feature_cols = [c for c in df_num.columns if c not in TARGET_VARS_ALL]
        feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
        for pat in PHYSICAL_EXCLUDE_PATTERNS:
            feature_cols = [c for c in feature_cols if pat not in c]

        # (H) 多重共線性・ゼロ分散除外
        v = df_num[feature_cols].var()
        feature_cols = v[v > 0].index.tolist()
        feature_cols = remove_collinear_features(df_num, feature_cols)

        if len(feature_cols) == 0:
            print(f"[WARN] No usable features for {file} - {target}")
            continue

        # -----------------------------
        # (I) Scaling (-1, 1)
        # -----------------------------
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_scaled = scaler.fit_transform(df_num[feature_cols])
        y = df_num[target].to_numpy()
        sample_ids = df_num.index.astype(str).to_numpy()

        # -----------------------------
        # Hyperparameter tuning
        # -----------------------------
        param_grid = {
            "n_estimators": [200, 500],
            "max_depth": [None, 20, 40],
            "min_samples_leaf": [1, 3, 5],
            "max_features": ["sqrt", 0.3, 0.5]
        }
        cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
        grid = GridSearchCV(
            estimator=RandomForestRegressor(random_state=SEED, n_jobs=-1),
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_scaled, y)
        best_model = grid.best_estimator_

        print(f"[BEST] {file} - {target} | params={grid.best_params_} | "
              f"mean_neg_RMSE={grid.best_score_:.6f}")

        cv_results = pd.DataFrame(grid.cv_results_)
        cv_results.to_csv(
            os.path.join(out_root, "tuning_results_RandomForest_Voc_oldFeatLogic.csv"),
            index=False
        )

        # -----------------------------
        # (K) foldID 付き CV10 OOF 予測
        # -----------------------------
        oof_pred = np.zeros_like(y, dtype=float)
        oof_fold = np.empty_like(y, dtype=object)

        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_scaled, y), start=1):
            X_tr, y_tr = X_scaled[train_idx], y[train_idx]
            X_te = X_scaled[test_idx]

            model_fold = RandomForestRegressor(**best_model.get_params())
            model_fold.fit(X_tr, y_tr)
            y_te_pred = model_fold.predict(X_te)

            oof_pred[test_idx] = y_te_pred
            oof_fold[test_idx] = f"Fold{fold_idx}"

        cv_r2   = safe_r2(y, oof_pred)
        cv_rmse = float(np.sqrt(mean_squared_error(y, oof_pred)))
        cv_mae  = float(mean_absolute_error(y, oof_pred))

        # -----------------------------
        # 結果保存用ディレクトリ
        # -----------------------------
        file_tag = file.replace(".csv", "")
        run_dir = os.path.join(out_root, f"{model_name_dir}_{file_tag}_{target}")
        os.makedirs(run_dir, exist_ok=True)

        # 1. モデル・スケーラ保存
        joblib.dump(best_model, os.path.join(run_dir, f"{file_tag}_{target}_model.joblib"))
        joblib.dump(scaler,     os.path.join(run_dir, f"{file_tag}_{target}_scaler.joblib"))

        # 2. foldID・誤差付き CV10_OOF 予測CSV
        df_pred_oof = pd.DataFrame({
            "SampleID": sample_ids,
            "foldID": oof_fold,
            "Observed": y.astype(float),
            "Predicted": oof_pred.astype(float)
        })
        df_pred_oof["Error"]   = df_pred_oof["Predicted"] - df_pred_oof["Observed"]
        df_pred_oof["AbsError"] = df_pred_oof["Error"].abs()
        df_pred_oof["Type"]    = "CV10_OOF"

        df_pred_oof, thr = flag_high_error(df_pred_oof, quantile_prob=0.9)

        df_pred_oof.to_csv(
            os.path.join(run_dir, f"{file_tag}_{target}_pred_OOF_withError.csv"),
            index=False
        )

        # 3. 特徴量付きテーブル
        df_feat = df_num[feature_cols].copy()
        df_feat["SampleID"] = sample_ids
        df_feat = df_feat.set_index("SampleID")

        df_pred_feat = df_pred_oof.set_index("SampleID").join(df_feat, how="left")
        df_pred_feat.to_csv(
            os.path.join(run_dir, f"{file_tag}_{target}_pred_OOF_withError_andFeatures.csv")
        )

        # 4. 高誤差サンプルだけ
        df_high = df_pred_feat[df_pred_feat["HighErrorFlag"]].copy()
        df_high.to_csv(
            os.path.join(run_dir, f"{file_tag}_{target}_highErrorSamples.csv")
        )

        # 5. 重要度
        imp_rows = compute_importance(best_model, feature_cols)
        for r in imp_rows:
            r.update({"File": file, "Target": target, "Model": model_name_dir})
            importance_rows.append(r)

        # 6. サマリー
        summary_rows.append({
            "File": file,
            "Target": target,
            "Model": model_name_dir,
            "CV10_R2": cv_r2,
            "CV10_RMSE": cv_rmse,
            "CV10_MAE": cv_mae,
            "n_samples": int(df_num.shape[0]),
            "n_features": int(len(feature_cols)),
            "Best_Params": json.dumps(grid.best_params_),
            "HighErrorThreshold": thr
        })

        print(f"  Target={target} | CV10_R2={cv_r2:.3f} | Best_Params={grid.best_params_}")

# -----------------------------
# 全体サマリーと重要度
# -----------------------------
if summary_rows:
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(out_root, "all_summary_CV10_RandomForest_Voc_oldFeatLogic.csv"),
        index=False
    )

if importance_rows:
    pd.DataFrame(importance_rows).to_csv(
        os.path.join(out_root, "all_importance_RF_Voc_oldFeatLogic.csv"),
        index=False
    )

print("\n[DONE] RandomForest Voc Outlier-aware Analysis (old feature-selection logic) completed.")



=== Processing Random Forest Voc (oldFeatLogic): m_base_FP_rebuilt.csv ===
[BEST] m_base_FP_rebuilt.csv - Voc | params={'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200} | mean_neg_RMSE=-0.079954
  Target=Voc | CV10_R2=0.723 | Best_Params={'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}

[DONE] RandomForest Voc Outlier-aware Analysis (old feature-selection logic) completed.
